---
## 0. Setup: GPU Check & Data Download

In [ ]:
import torch
import os

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

In [ ]:
# Download Tiny Shakespeare
!mkdir -p data weights
!wget -q -O data/shakespeare.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wc -c data/shakespeare.txt
!head -5 data/shakespeare.txt

---
## 1. Shared Foundation: Data, Vocab, Utilities

This cell sets up everything shared across all models.

In [ ]:
import os
import sys
import time
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Data Loading ----
with open('data/shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
MASK_CHAR = '_'
assert MASK_CHAR not in chars, f"MASK char '{MASK_CHAR}' already in text!"
chars_with_mask = [MASK_CHAR] + chars  # MASK at index 0
vocab_size = len(chars_with_mask)

stoi = {ch: i for i, ch in enumerate(chars_with_mask)}
itos = {i: ch for i, ch in enumerate(chars_with_mask)}
mask_token_id = stoi[MASK_CHAR]  # = 0

def encode(s):
    return [stoi[ch] for ch in s]

def decode(tokens):
    return ''.join([itos[t] for t in tokens])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'Vocab size: {vocab_size} ({vocab_size-1} chars + 1 MASK)')
print(f'Dataset: {len(data):,} tokens')
print(f'Train: {len(train_data):,}, Val: {len(val_data):,}')
print(f'Device: {device}')

---
## 2. Phase 3: Bidirectional Transformer Denoiser (step2)

Replace the MLP with a bidirectional transformer. This is the quality jump.

**Architecture:**
- Token embedding + RoPE (Rotary Positional Embeddings)
- Multi-head **bidirectional** self-attention (`is_causal=False`)
- MLP with ReluSquared activation
- RMSNorm (functional, no learnable params)
- Pre-norm residual connections

**Starting small** (build plan): 4 layers, 4 heads, 128 embd

In [ ]:
# ============================================================================
# Step 2: Hyperparameters (small model per build plan)
# ============================================================================
batch_size = 64
block_size = 256
max_iters_step2 = 10000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4

# Small model architecture
n_embd = 128
n_head = 4
n_layer = 4
head_dim = n_embd // n_head  # 32

print(f'Architecture: {n_layer}L / {n_head}H / {n_embd}E / {head_dim}D')
print(f'Block size: {block_size}, Batch size: {batch_size}')
print(f'Training for {max_iters_step2} iterations')

In [ ]:
# ============================================================================
# Batching with Masking
# ============================================================================
def get_batch_diffusion(split):
    """Get a batch with random masking for diffusion training.
    Each sample gets a random mask probability from [0,1],
    then each token is independently masked with that probability."""
    d = train_data if split == 'train' else val_data
    idx = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in idx])
    y = x.clone()  # clean tokens = targets

    # Random masking: uniform mask probability per sample
    mask_probs = torch.rand(batch_size, 1)
    mask = torch.rand(batch_size, block_size) < mask_probs
    x[mask] = mask_token_id

    x, y, mask = x.to(device), y.to(device), mask.to(device)
    return x, y, mask

# Quick test
xb, yb, mb = get_batch_diffusion('train')
print(f'Batch shapes: x={xb.shape}, y={yb.shape}, mask={mb.shape}')
print(f'Masked tokens in batch: {mb.sum().item()} / {mb.numel()} = {mb.float().mean():.2%}')

In [ ]:
# ============================================================================
# Transformer Components
# ============================================================================

def norm(x):
    """Functional RMSNorm — no learnable parameters."""
    return F.rms_norm(x, (x.size(-1),))


def apply_rotary_emb(x, cos, sin):
    """Apply rotary positional embeddings to queries/keys.
    RoPE encodes relative position by rotating pairs of dimensions."""
    assert x.ndim == 4  # (B, T, H, D)
    d = x.shape[3] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return torch.cat([y1, y2], 3).to(x.dtype)


class MultiHeadAttention(nn.Module):
    """Bidirectional multi-head self-attention with RoPE.
    KEY DIFFERENCE FROM GPT: is_causal=False"""
    def __init__(self):
        super().__init__()
        self.c_q = nn.Linear(n_embd, n_embd, bias=False)
        self.c_k = nn.Linear(n_embd, n_embd, bias=False)
        self.c_v = nn.Linear(n_embd, n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, cos_sin):
        B, T, C = x.size()
        q = self.c_q(x).view(B, T, n_head, head_dim)
        k = self.c_k(x).view(B, T, n_head, head_dim)
        v = self.c_v(x).view(B, T, n_head, head_dim)

        cos, sin = cos_sin
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        q, k = norm(q), norm(k)  # QK norm for stability

        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)  # (B,H,T,D)

        # BIDIRECTIONAL attention — the key difference from GPT!
        y = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.c_proj(y)


class MLP(nn.Module):
    """FFN with ReluSquared: ReLU(x)^2"""
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()
        return self.c_proj(x)


class Block(nn.Module):
    """Pre-norm transformer block with residual connections."""
    def __init__(self):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.mlp = MLP()

    def forward(self, x, cos_sin):
        x = x + self.attn(norm(x), cos_sin)
        x = x + self.mlp(norm(x))
        return x


class DiffusionModel(nn.Module):
    """Bidirectional Transformer for discrete diffusion denoising."""
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)

        # Precompute rotary embeddings
        self.rotary_seq_len = block_size * 2
        cos, sin = self._precompute_rope(self.rotary_seq_len)
        self.register_buffer('cos', cos, persistent=False)
        self.register_buffer('sin', sin, persistent=False)

        self.blocks = nn.ModuleList([Block() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _precompute_rope(self, seq_len, base=10000, device=None):
        if device is None:
            device = self.token_emb.weight.device
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32, device=device)
        inv_freq = 1.0 / (base ** (channel_range / head_dim))
        t = torch.arange(seq_len, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)
        cos, sin = freqs.cos(), freqs.sin()
        return cos[None, :, None, :], sin[None, :, None, :]

    def forward(self, idx, targets=None, mask=None):
        B, T = idx.size()
        x = self.token_emb(idx)
        x = norm(x)

        assert T <= self.cos.size(1)
        cos_sin = (self.cos[:, :T], self.sin[:, :T])

        for block in self.blocks:
            x = block(x, cos_sin)
        x = norm(x)

        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits_flat = logits.view(B * T, C)
        targets_flat = targets.view(B * T)

        if mask is not None:
            mask_flat = mask.view(B * T).float()
            loss = F.cross_entropy(logits_flat, targets_flat, reduction='none')
            loss = (loss * mask_flat).sum() / mask_flat.sum()
        else:
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

# Instantiate
model_step2 = DiffusionModel().to(device)
n_params = sum(p.numel() for p in model_step2.parameters())
print(f'Transformer parameters: {n_params:,} ({n_params/1e6:.2f}M)')

In [ ]:
# ============================================================================
# Training Utilities
# ============================================================================

@torch.no_grad()
def estimate_loss_diffusion(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y, M = get_batch_diffusion(split)
            _, loss = model(X, Y, M)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


@torch.no_grad()
def generate_sample_greedy(model, prompt_len=16, length=240):
    """Single-pass greedy denoising (not proper iterative sampling yet)."""
    model.eval()
    x = torch.full((1, min(length, block_size)), mask_token_id, dtype=torch.long, device=device)
    x[0, :prompt_len] = data[:prompt_len].to(device)
    logits, _ = model(x)
    preds = logits.argmax(dim=-1)
    result = x.clone()
    result[0, prompt_len:] = preds[0, prompt_len:]
    model.train()
    return decode(result[0].cpu().tolist())

In [ ]:
# ============================================================================
# Train Step 2: Small Transformer (4L/4H/128E)
# ============================================================================
print('Training Step 2: Bidirectional Transformer Denoiser')
print(f'Architecture: {n_layer}L / {n_head}H / {n_embd}E')
print('=' * 60)

optimizer = torch.optim.AdamW(model_step2.parameters(), lr=learning_rate)
log_step2 = []
start = time.time()

for iter in range(max_iters_step2):
    if iter % eval_interval == 0 or iter == max_iters_step2 - 1:
        losses = estimate_loss_diffusion(model_step2)
        elapsed = time.time() - start
        log_step2.append({
            'iter': iter,
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
            'time': elapsed,
        })
        print(f'step {iter:5d}: train {losses["train"]:.4f}, val {losses["val"]:.4f}, time {elapsed:.1f}s')

        if iter > 0:
            sample = generate_sample_greedy(model_step2)
            print(f'  Sample: {repr(sample[:100])}\n')

    xb, yb, mb = get_batch_diffusion('train')
    _, loss = model_step2(xb, yb, mb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - start
print(f'\nTotal training time: {total_time:.1f}s')
print(f'\nLoss comparison:')
print(f'  Random guessing: {math.log(vocab_size):.4f}')
print(f'  MLP ceiling:     ~3.31')
print(f'  Transformer:     {log_step2[-1]["train_loss"]:.4f} (train), {log_step2[-1]["val_loss"]:.4f} (val)')

In [ ]:
# Save step2 weights
torch.save(model_step2.state_dict(), 'weights/step2_transformer.pt')
print('Saved weights/step2_transformer.pt')

# Show final samples
print('\nFinal samples (single-pass greedy, not iterative):')
for i in range(3):
    torch.manual_seed(i * 42)
    s = generate_sample_greedy(model_step2)
    print(f'  Sample {i+1}: {repr(s[:120])}')

---
## 3. Phase 4: Iterative Parallel Unmasking (step3_sampling)

Now we implement **proper generation** — iterative confidence-based unmasking:
1. Start from all-MASK sequence
2. Get model predictions + confidence scores
3. Unmask the most confident positions first
4. Repeat until all positions are unmasked
5. For longer text: generate in blocks, use previous block as context

In [ ]:
# ============================================================================
# Iterative Confidence-Based Parallel Decoding
# ============================================================================

@torch.no_grad()
def generate_diffusion(model, max_new_tokens=2000, prompt_len=16,
                        temp=0.8, confidence_threshold=0.95, top_k=3):
    """
    Generate text using iterative parallel unmasking.

    Algorithm:
    1. Start with prompt + all MASKs
    2. Get model logits, compute confidence (sum of top-k probs)
    3. Unmask positions where confidence > threshold
    4. If none exceed threshold, unmask the single most confident position
    5. Repeat until no MASKs remain
    6. Slide to next block using last prompt_len tokens as context

    Args:
        model: trained diffusion model
        max_new_tokens: total tokens to generate
        prompt_len: tokens of context to seed from data
        temp: sampling temperature
        confidence_threshold: threshold for unmasking
        top_k: number of top candidates for confidence + sampling
    """
    model.eval()
    all_tokens = data[:prompt_len].tolist()
    total_steps = 0

    while len(all_tokens) - prompt_len < max_new_tokens:
        # How many tokens to generate this block
        block_len = min(240, prompt_len + max_new_tokens - len(all_tokens))

        # Initialize: context + MASKs
        x = torch.full((1, block_size), mask_token_id, dtype=torch.long, device=device)
        x[0, :prompt_len] = torch.tensor(all_tokens[-prompt_len:], device=device)

        # Track which positions still need decoding
        masked = torch.zeros(1, block_size, dtype=torch.bool, device=device)
        masked[0, prompt_len:prompt_len + block_len] = True

        # Iteratively decode
        while masked.any():
            total_steps += 1

            logits, _ = model(x)
            logits[::,mask_token_id] = -float('inf')  # never sample MASK token
            probs = F.softmax(logits / temp, dim=-1)
            top_k_probs, top_k_indices = torch.topk(probs, k=top_k, dim=-1)
            confidences = top_k_probs.sum(dim=-1)  # confidence = sum of top-k probs

            # Unmask high-confidence positions
            decode_mask = (confidences >= confidence_threshold) & masked
            if not decode_mask.any():
                # Force unmask the single most confident masked position
                masked_confidences = torch.where(
                    masked, confidences, torch.tensor(-float('inf'), device=device)
                )
                decode_mask.view(-1)[masked_confidences.argmax()] = True

            # Sample from top-k distribution
            top_k_probs_norm = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
            sampled_k = torch.multinomial(
                top_k_probs_norm.view(-1, top_k), 1
            ).view(1, block_size)
            sampled_tokens = torch.gather(
                top_k_indices, -1, sampled_k.unsqueeze(-1)
            ).squeeze(-1)

            # Update: place sampled tokens at decoded positions
            x = torch.where(decode_mask, sampled_tokens, x)
            masked = masked & ~decode_mask

        # Extract generated tokens for this block
        all_tokens.extend(x[0, prompt_len:prompt_len + block_len].tolist())

    tokens_generated = len(all_tokens) - prompt_len
    print(f'Generated {tokens_generated} tokens in {total_steps} steps')
    print(f'Avg decoded per step: {tokens_generated / total_steps:.2f}')
    return decode(all_tokens)

In [ ]:
# ============================================================================
# Test iterative sampling with the step2 model
# ============================================================================
print('Testing iterative parallel unmasking with step2 model...')
print('=' * 60)

start = time.time()
output = generate_diffusion(model_step2, max_new_tokens=500, temp=0.8,
                            confidence_threshold=0.95, top_k=3)
gen_time = time.time() - start
print(f'Generation time: {gen_time:.2f}s')
print(f'\nGenerated text:\n{output}')

---
## 4. Phase 5: Final Diffusion Model (scaled up)

Now we scale up to match the reference repo:
- **6 layers, 6 heads, 384 embd** (~10.7M params)
- **10,000 iterations** (diffusion needs more since only masked tokens contribute to loss)

We need to redefine the model classes with the new hyperparams.

In [ ]:
# ============================================================================
# Final Model Hyperparameters (scaled up)
# ============================================================================

# Clear step2 model to free GPU memory
del model_step2, optimizer
torch.cuda.empty_cache() if device == 'cuda' else None

batch_size = 64
block_size = 256
max_iters_final = 10000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4

# Scaled up architecture (matching reference repo)
n_embd = 384
n_head = 6
n_layer = 6
head_dim = n_embd // n_head  # 64

print(f'Final architecture: {n_layer}L / {n_head}H / {n_embd}E / {head_dim}D')
print(f'Training for {max_iters_final} iterations')

In [ ]:
# ============================================================================
# Redefine model classes with new hyperparams
# (Python closures capture the module-level variables)
# ============================================================================

class MultiHeadAttentionFinal(nn.Module):
    def __init__(self, is_causal=False):
        super().__init__()
        self.is_causal = is_causal
        self.c_q = nn.Linear(n_embd, n_embd, bias=False)
        self.c_k = nn.Linear(n_embd, n_embd, bias=False)
        self.c_v = nn.Linear(n_embd, n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, cos_sin):
        B, T, C = x.size()
        q = self.c_q(x).view(B, T, n_head, head_dim)
        k = self.c_k(x).view(B, T, n_head, head_dim)
        v = self.c_v(x).view(B, T, n_head, head_dim)
        cos, sin = cos_sin
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        q, k = norm(q), norm(k)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=self.is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.c_proj(y)


class MLPFinal(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()
        return self.c_proj(x)


class BlockFinal(nn.Module):
    def __init__(self, is_causal=False):
        super().__init__()
        self.attn = MultiHeadAttentionFinal(is_causal=is_causal)
        self.mlp = MLPFinal()

    def forward(self, x, cos_sin):
        x = x + self.attn(norm(x), cos_sin)
        x = x + self.mlp(norm(x))
        return x


class FinalModel(nn.Module):
    """Shared architecture for both Diffusion and GPT models.
    The only difference is is_causal in attention."""
    def __init__(self, is_causal=False, use_mask_loss=True):
        super().__init__()
        self.is_causal = is_causal
        self.use_mask_loss = use_mask_loss

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.rotary_seq_len = block_size * 2
        cos, sin = self._precompute_rope(self.rotary_seq_len)
        self.register_buffer('cos', cos, persistent=False)
        self.register_buffer('sin', sin, persistent=False)

        self.blocks = nn.ModuleList([BlockFinal(is_causal=is_causal) for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _precompute_rope(self, seq_len, base=10000, device=None):
        if device is None:
            device = self.token_emb.weight.device
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32, device=device)
        inv_freq = 1.0 / (base ** (channel_range / head_dim))
        t = torch.arange(seq_len, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)
        cos, sin = freqs.cos(), freqs.sin()
        return cos[None, :, None, :], sin[None, :, None, :]

    def forward(self, idx, targets=None, mask=None):
        B, T = idx.size()
        x = self.token_emb(idx)
        x = norm(x)

        assert T <= self.cos.size(1)
        cos_sin = (self.cos[:, :T], self.sin[:, :T])

        for block in self.blocks:
            x = block(x, cos_sin)
        x = norm(x)

        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits_flat = logits.view(B * T, C)
        targets_flat = targets.view(B * T)

        if mask is not None and self.use_mask_loss:
            mask_flat = mask.view(B * T).float()
            loss = F.cross_entropy(logits_flat, targets_flat, reduction='none')
            loss = (loss * mask_flat).sum() / mask_flat.sum()
        else:
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss


print('Model classes defined (FinalModel supports both diffusion & GPT modes)')

In [ ]:
# ============================================================================
# Train Final Diffusion Model (6L/6H/384E, ~10.7M params)
# ============================================================================
print('Training Final Diffusion Model')
print('=' * 60)

diffusion_model = FinalModel(is_causal=False, use_mask_loss=True).to(device)
n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f'Parameters: {n_params:,} ({n_params/1e6:.1f}M)')

optimizer_diff = torch.optim.AdamW(diffusion_model.parameters(), lr=learning_rate)
log_diffusion = []
start = time.time()

for iter in range(max_iters_final):
    if iter % eval_interval == 0 or iter == max_iters_final - 1:
        losses = estimate_loss_diffusion(diffusion_model)
        elapsed = time.time() - start
        log_diffusion.append({
            'iter': iter,
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
            'time': elapsed,
        })
        print(f'step {iter:5d}: train {losses["train"]:.4f}, val {losses["val"]:.4f}, time {elapsed:.1f}s')

        if iter > 0 and iter % 2000 == 0:
            sample = generate_diffusion(diffusion_model, max_new_tokens=240,
                                        temp=0.8, confidence_threshold=0.95, top_k=2)
            print(f'  Sample: {repr(sample[:120])}\n')

    xb, yb, mb = get_batch_diffusion('train')
    _, loss = diffusion_model(xb, yb, mb)
    optimizer_diff.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_diff.step()

total_time = time.time() - start
print(f'\nTotal training time: {total_time:.1f}s')
print(f'Final loss: train {log_diffusion[-1]["train_loss"]:.4f}, val {log_diffusion[-1]["val_loss"]:.4f}')

In [ ]:
# Save diffusion weights
torch.save(diffusion_model.state_dict(), 'weights/diffusion.pt')
print('Saved weights/diffusion.pt')

# Generate a proper sample
print('\nFinal diffusion generation (2000 chars):')
print('=' * 60)
start = time.time()
output = generate_diffusion(diffusion_model, max_new_tokens=2000, temp=0.8,
                            confidence_threshold=0.95, top_k=2)
print(f'Generation time: {time.time() - start:.2f}s')
print(f'\n{output}')

---
## 5. Phase 5 (cont): GPT Baseline Model

Same architecture as diffusion, but with:
1. **Causal attention** (`is_causal=True`) — can only look left
2. **Next-token prediction** — targets are shifted by 1
3. **Sequential generation** — one token at a time
4. **All tokens** contribute to loss (not just masked)
5. **5,000 iterations** (half of diffusion since every token contributes to loss)

In [ ]:
# ============================================================================
# GPT Batching (next-token prediction, no masking)
# ============================================================================

def get_batch_gpt(split):
    """Standard autoregressive batching: x[i] -> y[i] = x[i+1]"""
    d = train_data if split == 'train' else val_data
    idx = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in idx])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in idx])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss_gpt(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_gpt(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


@torch.no_grad()
def generate_gpt(model, max_new_tokens=2000, prompt_len=16, temp=0.8):
    """Standard autoregressive generation: one token at a time."""
    model.eval()
    x = data[:prompt_len].unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        cur_context = x[:, -block_size:]
        logits, _ = model(cur_context)
        logits = logits[:, -1, :]  # last position only
        probs = F.softmax(logits / temp, dim=-1)
        next_token = (
            torch.argmax(probs, dim=-1, keepdim=True) if temp == 0
            else torch.multinomial(probs, num_samples=1)
        )
        x = torch.cat((x, next_token), dim=1)

    model.train()
    return decode(x[0].cpu().tolist())

print('GPT utilities defined')

In [ ]:
# ============================================================================
# Train GPT Model (5000 iters — half of diffusion)
# ============================================================================
max_iters_gpt = 5000

print('Training GPT Baseline Model')
print(f'Architecture: {n_layer}L / {n_head}H / {n_embd}E (same as diffusion)')
print(f'Key diff: causal attention, next-token prediction, {max_iters_gpt} iters')
print('=' * 60)

# GPT uses the same architecture but with causal attention
# Note: GPT vocab doesn't need MASK token, but we keep it for fair param comparison
gpt_model = FinalModel(is_causal=True, use_mask_loss=False).to(device)
n_params_gpt = sum(p.numel() for p in gpt_model.parameters())
print(f'Parameters: {n_params_gpt:,} ({n_params_gpt/1e6:.1f}M)')

optimizer_gpt = torch.optim.AdamW(gpt_model.parameters(), lr=learning_rate)
log_gpt = []
start = time.time()

for iter in range(max_iters_gpt):
    if iter % eval_interval == 0 or iter == max_iters_gpt - 1:
        losses = estimate_loss_gpt(gpt_model)
        elapsed = time.time() - start
        log_gpt.append({
            'iter': iter,
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
            'time': elapsed,
        })
        print(f'step {iter:5d}: train {losses["train"]:.4f}, val {losses["val"]:.4f}, time {elapsed:.1f}s')

        if iter > 0 and iter % 1000 == 0:
            sample = generate_gpt(gpt_model, max_new_tokens=240)
            print(f'  Sample: {repr(sample[:120])}\n')

    xb, yb = get_batch_gpt('train')
    _, loss = gpt_model(xb, yb)
    optimizer_gpt.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_gpt.step()

total_time = time.time() - start
print(f'\nTotal training time: {total_time:.1f}s')
print(f'Final loss: train {log_gpt[-1]["train_loss"]:.4f}, val {log_gpt[-1]["val_loss"]:.4f}')

In [ ]:
# Save GPT weights
torch.save(gpt_model.state_dict(), 'weights/gpt.pt')
print('Saved weights/gpt.pt')

# Generate a proper sample
print('\nFinal GPT generation (2000 chars):')
print('=' * 60)
start = time.time()
output_gpt = generate_gpt(gpt_model, max_new_tokens=2000, temp=0.8)
print(f'Generation time: {time.time() - start:.2f}s')
print(f'\n{output_gpt}')

---
## 6. Comparison: Diffusion vs GPT

In [ ]:
# ============================================================================
# Side-by-Side Comparison
# ============================================================================

print('=' * 70)
print('MODEL COMPARISON')
print('=' * 70)

n_diff = sum(p.numel() for p in diffusion_model.parameters())
n_gpt = sum(p.numel() for p in gpt_model.parameters())

print(f'\n{"":20s} {"Diffusion":>15s} {"GPT":>15s}')
print(f'{"─"*50}')
print(f'{"Parameters":20s} {n_diff/1e6:>14.1f}M {n_gpt/1e6:>14.1f}M')
print(f'{"Attention":20s} {"Bidirectional":>15s} {"Causal":>15s}')
print(f'{"Training iters":20s} {max_iters_final:>15,d} {max_iters_gpt:>15,d}')
print(f'{"Final train loss":20s} {log_diffusion[-1]["train_loss"]:>15.4f} {log_gpt[-1]["train_loss"]:>15.4f}')
print(f'{"Final val loss":20s} {log_diffusion[-1]["val_loss"]:>15.4f} {log_gpt[-1]["val_loss"]:>15.4f}')
print(f'{"Training time":20s} {log_diffusion[-1]["time"]:>14.1f}s {log_gpt[-1]["time"]:>14.1f}s')
print(f'{"Generation":20s} {"Parallel blocks":>15s} {"Sequential L→R":>15s}')
print(f'{"Loss scope":20s} {"Masked only":>15s} {"All tokens":>15s}')

print('\n' + '=' * 70)
print('5 KEY CHANGES from GPT → Diffusion:')
print('  1. Add MASK token to vocabulary')
print('  2. Bidirectional attention (is_causal=False)')
print('  3. Confidence-based parallel decoding (not sequential)')
print('  4. Denoising objective (not next-token prediction)')
print('  5. Loss on masked tokens only (not all tokens)')
print('=' * 70)

In [ ]:
# ============================================================================
# Generate comparison samples
# ============================================================================

print('DIFFUSION OUTPUT (500 chars):')
print('-' * 60)
torch.manual_seed(42)
diff_out = generate_diffusion(diffusion_model, max_new_tokens=500, temp=0.8,
                               confidence_threshold=0.95, top_k=2)
print(diff_out)

print('\n\nGPT OUTPUT (500 chars):')
print('-' * 60)
torch.manual_seed(42)
gpt_out = generate_gpt(gpt_model, max_new_tokens=500, temp=0.8)
print(gpt_out)

---
## 7. Download Weights

Run this to download the trained weights to your local machine.

In [ ]:
# Download weights from Colab
try:
    from google.colab import files
    print('Downloading weights...')
    files.download('weights/diffusion.pt')
    files.download('weights/gpt.pt')
    files.download('weights/step2_transformer.pt')
    print('Done! Move these to your local weights/ folder.')
except ImportError:
    print('Not running on Colab. Weights saved to weights/ directory.')
    print('Files:')
    import glob
    for f in glob.glob('weights/*.pt'):
        size_mb = os.path.getsize(f) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')

---
## 8. Training Log Summary

Copy these numbers to your build log.

In [ ]:
import json

summary = {
    'step2_transformer_small': {
        'architecture': f'{4}L/{4}H/{128}E',
        'log': log_step2,
    },
    'diffusion_final': {
        'architecture': f'{n_layer}L/{n_head}H/{n_embd}E',
        'log': log_diffusion,
    },
    'gpt_final': {
        'architecture': f'{n_layer}L/{n_head}H/{n_embd}E',
        'log': log_gpt,
    },
}

with open('training_log.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Training log saved to training_log.json')
print('\nSummary:')
for name, info in summary.items():
    final = info['log'][-1]
    print(f'  {name} ({info["architecture"]}): '
          f'train={final["train_loss"]:.4f}, val={final["val_loss"]:.4f}, '
          f'time={final["time"]:.1f}s')

# Also download the log
try:
    from google.colab import files
    files.download('training_log.json')
except ImportError:
    pass